# 03 · Modeling, evaluation and restaurant intelligence app

This notebook uses the outputs of notebooks 1 and 2. It compares a majority class baseline, Logistic Regression and LinearSVC with bilingual TF-IDF text features. Restaurants are separated between development and final test. The final test is evaluated once. The saved pipeline feeds a separate Streamlit app and a FastAPI endpoint. MLflow records the experiments.

**Important:** The sentiment target comes from star ratings. It is a rating based label, not an independent annotation of emotions or service issues. Historical alerts end in May 2023.

Run from the project root, with `data/`, `app/`, and `artifacts/` at the same level as this notebook. Install `requirements-modeling.txt` before running.

In [1]:
from datetime import datetime, timezone
from functools import lru_cache
from html import escape
from os import getenv
from pathlib import Path
from time import perf_counter, sleep
from urllib.request import urlopen


import ast
import gc
import json
import os
import platform
import signal
import subprocess
import sys
import warnings
warnings.filterwarnings("ignore")

import joblib
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
import plotly.express as px
import sklearn
import streamlit as st

from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from IPython.display import HTML, display
from lime.lime_text import LimeTextExplainer
from mlflow import MlflowClient
from mlflow.models import infer_signature
from pydantic import BaseModel, Field
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    recall_score
)
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# Define project paths
ROOT_PATH = Path.cwd().resolve()
DATA_PATH = ROOT_PATH / 'data'
ARTIFACTS_PATH = ROOT_PATH / 'artifacts'

ARTIFACTS_PATH.mkdir(
    parents=True,
    exist_ok=True
)

# Set the random seed
RANDOM_STATE = 42


In [2]:
# Record package versions
environment_summary = pd.Series(
    {
        'python': sys.version.split()[0],
        'operating_system': platform.platform(),
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scikit_learn': sklearn.__version__
    },
    name='version'
)

display(environment_summary.to_frame())

,version
python,3.13.5
operating_system,macOS-27.0-arm64-arm-64bit-Mach-O
numpy,2.2.4
pandas,2.2.3
scikit_learn,1.6.1


## 1. Load and validate the prepared corpus


In [3]:
# Define input paths
modeling_corpus_path = (
    DATA_PATH / 'modeling_corpus.parquet'
)

project_config_path = (
    DATA_PATH / 'project_config.json'
)

# Load methodology settings
with project_config_path.open(
    mode='r',
    encoding='utf-8'
) as config_file:
    project_config = json.load(config_file)

# Load the modeling corpus
modeling_corpus = pd.read_parquet(
    modeling_corpus_path,
    engine='pyarrow'
)
# Check that this notebook uses the English classes from notebook 2.
expected_classes = ['Negative', 'Neutral', 'Positive']
assert project_config['sentiment']['classes'] == expected_classes
assert set(modeling_corpus['sentiment'].dropna().unique()) == set(expected_classes)
assert modeling_corpus['review_id'].is_unique
assert modeling_corpus['review_text'].notna().all()
assert set(modeling_corpus['dataset_split']) == {'train', 'validation', 'test'}
print('English corpus and grouped split validated.')


English corpus and grouped split validated.


In [4]:
# Validate the loaded corpus
corpus_validation = pd.Series(
    {
        'rows': len(modeling_corpus),
        'columns': modeling_corpus.shape[1],
        'unique_reviews': (
            modeling_corpus['review_id'].nunique()
        ),
        'unique_restaurants': (
            modeling_corpus['item_id'].nunique()
        ),
        'missing_values': (
            modeling_corpus.isna().sum().sum()
        ),
        'duplicated_review_ids': (
            modeling_corpus['review_id']
            .duplicated()
            .sum()
        ),
        'train_reviews': (
            modeling_corpus['dataset_split']
            .eq('train')
            .sum()
        ),
        'validation_reviews': (
            modeling_corpus['dataset_split']
            .eq('validation')
            .sum()
        ),
        'test_reviews': (
            modeling_corpus['dataset_split']
            .eq('test')
            .sum()
        )
    },
    name='result'
)

display(corpus_validation.to_frame())

,result
rows,1157800
columns,7
unique_reviews,1157800
unique_restaurants,10056
missing_values,0
duplicated_review_ids,0
train_reviews,931880
validation_reviews,115008
test_reviews,110912


## 2. Set up training and validation


In [5]:
# Define the class order
sentiment_order = [
    'Negative',
    'Neutral',
    'Positive'
]

# Select training and validation sets
train_mask = (
    modeling_corpus['dataset_split']
    .eq('train')
)

validation_mask = (
    modeling_corpus['dataset_split']
    .eq('validation')
)

# Separate input text and target labels
x_train = modeling_corpus.loc[
    train_mask,
    'review_text'
]

y_train = modeling_corpus.loc[
    train_mask,
    'sentiment'
]

x_validation = modeling_corpus.loc[
    validation_mask,
    'review_text'
]

y_validation = modeling_corpus.loc[
    validation_mask,
    'sentiment'
]

print('Training:', f'{len(x_train):,}')
print('Validation:', f'{len(x_validation):,}')

Training: 931,880
Validation: 115,008


## 3. Evaluation functions


In [6]:
# Store experiment results
experiment_results = {}


def evaluate_predictions(
    model_name,
    y_true,
    y_predicted,
    training_seconds
):
    """
    Calculate the main classification metrics.
    """

    metrics = {
        'accuracy': accuracy_score(
            y_true,
            y_predicted
        ),
        'balanced_accuracy': balanced_accuracy_score(
            y_true,
            y_predicted
        ),
        'macro_f1': f1_score(
            y_true,
            y_predicted,
            average='macro',
            zero_division=0
        ),
        'weighted_f1': f1_score(
            y_true,
            y_predicted,
            average='weighted',
            zero_division=0
        ),
        'negative_recall': recall_score(
            y_true,
            y_predicted,
            labels=['Negative'],
            average=None,
            zero_division=0
        )[0],
        'training_seconds': training_seconds
    }

    experiment_results[model_name] = metrics

    metrics_table = (
        pd.Series(
            metrics,
            name=model_name
        )
        .to_frame('value')
        .round(4)
    )

    report_table = (
        pd.DataFrame(
            classification_report(
                y_true,
                y_predicted,
                labels=sentiment_order,
                output_dict=True,
                zero_division=0
            )
        )
        .T
        .round(4)
    )

    confusion_table = pd.DataFrame(
        confusion_matrix(
            y_true,
            y_predicted,
            labels=sentiment_order
        ),
        index=[
            f'Real: {label}'
            for label in sentiment_order
        ],
        columns=[
            f'Predicted: {label}'
            for label in sentiment_order
        ]
    )

    return (
        metrics_table,
        report_table,
        confusion_table
    )

## 4. Majority class baseline


In [7]:
# Create dummy input features
dummy_x_train = np.zeros(
    (len(y_train), 1),
    dtype='int8'
)

dummy_x_validation = np.zeros(
    (len(y_validation), 1),
    dtype='int8'
)

# Create and train the baseline model
dummy_model = DummyClassifier(
    strategy='most_frequent'
)

start_time = perf_counter()

dummy_model.fit(
    dummy_x_train,
    y_train
)

dummy_training_seconds = (
    perf_counter() - start_time
)

# Predict on the validation set
dummy_predictions = dummy_model.predict(
    dummy_x_validation
)

In [8]:
(
    dummy_metrics,
    dummy_report,
    dummy_confusion
) = evaluate_predictions(
    model_name='dummy_most_frequent',
    y_true=y_validation,
    y_predicted=dummy_predictions,
    training_seconds=dummy_training_seconds
)

display(dummy_metrics)
display(dummy_report)
display(dummy_confusion)

,value
accuracy,0.7569
balanced_accuracy,0.3333
macro_f1,0.2872
weighted_f1,0.6522
negative_recall,0.0000
training_seconds,0.2331


,precision,recall,f1-score,support
Negative,0.0000,0.0000,0.0000,15335.0000
Neutral,0.0000,0.0000,0.0000,12623.0000
Positive,0.7569,1.0000,0.8616,87050.0000
accuracy,0.7569,0.7569,0.7569,0.7569
macro avg,0.2523,0.3333,0.2872,115008.0000
weighted avg,0.5729,0.7569,0.6522,115008.0000


,Predicted: Negative,Predicted: Neutral,Predicted: Positive
Real: Negative,0,0,15335
Real: Neutral,0,0,12623
Real: Positive,0,0,87050


## 5. Bilingual TF-IDF features


In [9]:
# Create one vectorizer for both models
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents=None,
    stop_words=None,
    token_pattern=r'(?u)\b\w\w+\b',
    ngram_range=(1, 2),
    min_df=10,
    max_df=0.98,
    max_features=60_000,
    sublinear_tf=True,
    norm='l2',
    dtype=np.float32
)

# Fit TF-IDF on the training set only
start_time = perf_counter()

x_train_tfidf = (
    tfidf_vectorizer
    .fit_transform(x_train)
)

tfidf_fit_seconds = (
    perf_counter() - start_time
)

# Transform validation text
start_time = perf_counter()

x_validation_tfidf = (
    tfidf_vectorizer
    .transform(x_validation)
)

tfidf_validation_seconds = (
    perf_counter() - start_time
)

In [10]:
# Measure sparse matrix memory use
def sparse_matrix_size_mb(matrix):
    return (
        matrix.data.nbytes
        + matrix.indices.nbytes
        + matrix.indptr.nbytes
    ) / 1024 ** 2


tfidf_validation = pd.Series(
    {
        'train_rows': x_train_tfidf.shape[0],
        'validation_rows': (
            x_validation_tfidf.shape[0]
        ),
        'features': x_train_tfidf.shape[1],
        'train_non_zero_values': (
            x_train_tfidf.nnz
        ),
        'validation_non_zero_values': (
            x_validation_tfidf.nnz
        ),
        'train_density_percentage': (
            x_train_tfidf.nnz
            / (
                x_train_tfidf.shape[0]
                * x_train_tfidf.shape[1]
            )
            * 100
        ),
        'train_matrix_size_mb': (
            sparse_matrix_size_mb(
                x_train_tfidf
            )
        ),
        'validation_matrix_size_mb': (
            sparse_matrix_size_mb(
                x_validation_tfidf
            )
        ),
        'fit_transform_seconds': (
            tfidf_fit_seconds
        ),
        'validation_transform_seconds': (
            tfidf_validation_seconds
        )
    },
    name='result'
)

display(
    tfidf_validation
    .to_frame()
    .round(4)
)

,result
train_rows,931880.0000
validation_rows,115008.0000
features,60000.0000
train_non_zero_values,75910459.0000
validation_non_zero_values,9577860.0000
train_density_percentage,0.1358
train_matrix_size_mb,582.7057
validation_matrix_size_mb,73.5120
fit_transform_seconds,48.4412
validation_transform_seconds,4.4841


## 6. Logistic Regression baseline


In [11]:
# Configure Logistic Regression
logistic_model = LogisticRegression(
    C=1.0,
    penalty='l2',
    solver='saga',
    class_weight='balanced',
    max_iter=100,
    tol=1e-3,
    random_state=RANDOM_STATE
)

# Train the model
start_time = perf_counter()

logistic_model.fit(
    x_train_tfidf,
    y_train
)

logistic_training_seconds = (
    perf_counter() - start_time
)

# Predict on the validation set
start_time = perf_counter()

logistic_predictions = logistic_model.predict(
    x_validation_tfidf
)

logistic_prediction_seconds = (
    perf_counter() - start_time
)

In [12]:
# Keep the number of completed iterations
previous_iterations_used = int(
    logistic_model.n_iter_.max()
)

# Continue training from the fitted coefficients
logistic_model.set_params(
    warm_start=True,
    max_iter=300
)

with warnings.catch_warnings(
    record=True
) as caught_warnings:

    warnings.simplefilter(
        'always',
        ConvergenceWarning
    )

    start_time = perf_counter()

    logistic_model.fit(
        x_train_tfidf,
        y_train
    )

    logistic_continuation_seconds = (
        perf_counter() - start_time
    )

# Check whether the warning appeared again
convergence_warning_detected = any(
    issubclass(
        warning.category,
        ConvergenceWarning
    )
    for warning in caught_warnings
)

continuation_iterations_used = int(
    logistic_model.n_iter_.max()
)

# Update total training time
logistic_training_seconds += (
    logistic_continuation_seconds
)

# Update the predictions
start_time = perf_counter()

logistic_predictions = logistic_model.predict(
    x_validation_tfidf
)

logistic_prediction_seconds = (
    perf_counter() - start_time
)

# Show convergence status
logistic_convergence_summary = pd.Series(
    {
        'previous_iterations': (
            previous_iterations_used
        ),
        'continuation_iterations': (
            continuation_iterations_used
        ),
        'estimated_total_iterations': (
            previous_iterations_used
            + continuation_iterations_used
        ),
        'additional_training_seconds': (
            logistic_continuation_seconds
        ),
        'total_training_seconds': (
            logistic_training_seconds
        ),
        'convergence_warning_detected': (
            convergence_warning_detected
        ),
        'converged': (
            not convergence_warning_detected
        )
    },
    name='result'
)

display(
    logistic_convergence_summary
    .to_frame()
    .round(4)
)

,result
previous_iterations,100
continuation_iterations,300
estimated_total_iterations,400
additional_training_seconds,282.4674
total_training_seconds,377.0120
convergence_warning_detected,True
converged,False


### Logistic Regression: validation


In [13]:
(
    logistic_metrics,
    logistic_report,
    logistic_confusion
) = evaluate_predictions(
    model_name='logistic_regression_tfidf',
    y_true=y_validation,
    y_predicted=logistic_predictions,
    training_seconds=logistic_training_seconds
)

display(logistic_metrics)
display(logistic_report)
display(logistic_confusion)

,value
accuracy,0.8820
balanced_accuracy,0.8260
macro_f1,0.7890
weighted_f1,0.8900
negative_recall,0.8889
training_seconds,377.0120


,precision,recall,f1-score,support
Negative,0.8154,0.8889,0.8506,15335.0000
Neutral,0.4965,0.6789,0.5736,12623.0000
Positive,0.9778,0.9102,0.9428,87050.0000
accuracy,0.8820,0.8820,0.8820,0.8820
macro avg,0.7633,0.8260,0.7890,115008.0000
weighted avg,0.9033,0.8820,0.8900,115008.0000


,Predicted: Negative,Predicted: Neutral,Predicted: Positive
Real: Negative,13631,1608,96
Real: Neutral,2351,8570,1702
Real: Positive,734,7083,79233


In [14]:
validation_model_comparison = (
    pd.DataFrame(experiment_results)
    .T
    .sort_values(
        by='macro_f1',
        ascending=False
    )
    .round(4)
)

display(validation_model_comparison)

,accuracy,balanced_accuracy,macro_f1,weighted_f1,negative_recall,training_seconds
logistic_regression_tfidf,0.8820,0.8260,0.7890,0.8900,0.8889,377.0120
dummy_most_frequent,0.7569,0.3333,0.2872,0.6522,0.0000,0.2331


## 7. LinearSVC baseline


In [15]:
# Configure the linear SVM classifier
linear_svc_model = LinearSVC(
    C=1.0,
    penalty='l2',
    loss='squared_hinge',
    class_weight='balanced',
    dual='auto',
    max_iter=5_000,
    tol=1e-3,
    random_state=RANDOM_STATE
)

# Train and capture warnings
with warnings.catch_warnings(
    record=True
) as linear_svc_warnings:

    warnings.simplefilter(
        'always',
        ConvergenceWarning
    )

    start_time = perf_counter()

    linear_svc_model.fit(
        x_train_tfidf,
        y_train
    )

    linear_svc_training_seconds = (
        perf_counter() - start_time
    )

# Check convergence
linear_svc_warning_detected = any(
    issubclass(
        warning.category,
        ConvergenceWarning
    )
    for warning in linear_svc_warnings
)

linear_svc_iterations = int(
    np.max(
        np.atleast_1d(
            linear_svc_model.n_iter_
        )
    )
)

# Predict on the validation set
start_time = perf_counter()

linear_svc_predictions = linear_svc_model.predict(
    x_validation_tfidf
)

linear_svc_prediction_seconds = (
    perf_counter() - start_time
)

# Summarize the training run
linear_svc_training_summary = pd.Series(
    {
        'iterations': linear_svc_iterations,
        'training_seconds': (
            linear_svc_training_seconds
        ),
        'prediction_seconds': (
            linear_svc_prediction_seconds
        ),
        'convergence_warning_detected': (
            linear_svc_warning_detected
        ),
        'converged': (
            not linear_svc_warning_detected
        )
    },
    name='result'
)

display(
    linear_svc_training_summary
    .to_frame()
    .round(4)
)

,result
iterations,17
training_seconds,62.9298
prediction_seconds,0.0490
convergence_warning_detected,False
converged,True


In [16]:
(
    linear_svc_metrics,
    linear_svc_report,
    linear_svc_confusion
) = evaluate_predictions(
    model_name='linear_svc_tfidf',
    y_true=y_validation,
    y_predicted=linear_svc_predictions,
    training_seconds=linear_svc_training_seconds
)

display(linear_svc_metrics)
display(linear_svc_report)
display(linear_svc_confusion)

,value
accuracy,0.9007
balanced_accuracy,0.8113
macro_f1,0.8004
weighted_f1,0.9028
negative_recall,0.8659
training_seconds,62.9298


,precision,recall,f1-score,support
Negative,0.8368,0.8659,0.8511,15335.0000
Neutral,0.5682,0.6207,0.5933,12623.0000
Positive,0.9663,0.9475,0.9568,87050.0000
accuracy,0.9007,0.9007,0.9007,0.9007
macro avg,0.7904,0.8113,0.8004,115008.0000
weighted avg,0.9054,0.9007,0.9028,115008.0000


,Predicted: Negative,Predicted: Neutral,Predicted: Positive
Real: Negative,13278,1849,208
Real: Neutral,2122,7835,2666
Real: Positive,468,4106,82476


## 8. Tune LinearSVC on validation only


In [17]:
# Define regularization values
svc_c_values = [
    0.1,
    0.5,
    1.0,
    2.0
]

svc_tuning_rows = []
svc_candidate_models = {}
svc_candidate_predictions = {}

for c_value in svc_c_values:

    candidate_model = LinearSVC(
        C=c_value,
        penalty='l2',
        loss='squared_hinge',
        class_weight='balanced',
        dual='auto',
        max_iter=5_000,
        tol=1e-3,
        random_state=RANDOM_STATE
    )

    with warnings.catch_warnings(
        record=True
    ) as candidate_warnings:

        warnings.simplefilter(
            'always',
            ConvergenceWarning
        )

        start_time = perf_counter()

        candidate_model.fit(
            x_train_tfidf,
            y_train
        )

        training_seconds = (
            perf_counter() - start_time
        )

    start_time = perf_counter()

    candidate_predictions = (
        candidate_model.predict(
            x_validation_tfidf
        )
    )

    prediction_seconds = (
        perf_counter() - start_time
    )

    warning_detected = any(
        issubclass(
            warning.category,
            ConvergenceWarning
        )
        for warning in candidate_warnings
    )

    svc_candidate_models[c_value] = (
        candidate_model
    )

    svc_candidate_predictions[c_value] = (
        candidate_predictions
    )

    svc_tuning_rows.append(
        {
            'c_value': c_value,
            'accuracy': accuracy_score(
                y_validation,
                candidate_predictions
            ),
            'balanced_accuracy': (
                balanced_accuracy_score(
                    y_validation,
                    candidate_predictions
                )
            ),
            'macro_f1': f1_score(
                y_validation,
                candidate_predictions,
                average='macro',
                zero_division=0
            ),
            'weighted_f1': f1_score(
                y_validation,
                candidate_predictions,
                average='weighted',
                zero_division=0
            ),
            'negative_recall': recall_score(
                y_validation,
                candidate_predictions,
                labels=['Negative'],
                average=None,
                zero_division=0
            )[0],
            'iterations': int(
                np.max(
                    np.atleast_1d(
                        candidate_model.n_iter_
                    )
                )
            ),
            'training_seconds': (
                training_seconds
            ),
            'prediction_seconds': (
                prediction_seconds
            ),
            'converged': (
                not warning_detected
            )
        }
    )

svc_tuning_table = (
    pd.DataFrame(svc_tuning_rows)
    .sort_values(
        by=[
            'macro_f1',
            'negative_recall'
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    svc_tuning_table.round(4)
)

,c_value,accuracy,balanced_accuracy,macro_f1,weighted_f1,negative_recall,iterations,training_seconds,prediction_seconds,converged
0,0.1000,0.9072,0.8206,0.8116,0.9085,0.8807,13,34.3952,0.0522,True
1,0.5000,0.9033,0.8150,0.8047,0.9051,0.8705,18,54.9741,0.0498,True
2,1.0000,0.9007,0.8113,0.8004,0.9028,0.8659,17,62.2853,0.0500,True
3,2.0000,0.8980,0.8078,0.7961,0.9004,0.8595,18,64.0552,0.0522,True


### Refine regularization values


In [18]:
# Search around the best regularization value
refinement_c_values = [
    0.01,
    0.03,
    0.05,
    0.20
]

# Remove duplicate results
svc_tuning_rows = [
    row
    for row in svc_tuning_rows
    if float(row['c_value'])
    not in refinement_c_values
]

for c_value in refinement_c_values:

    candidate_model = LinearSVC(
        C=c_value,
        penalty='l2',
        loss='squared_hinge',
        class_weight='balanced',
        dual='auto',
        max_iter=5_000,
        tol=1e-3,
        random_state=RANDOM_STATE
    )

    with warnings.catch_warnings(
        record=True
    ) as candidate_warnings:

        warnings.simplefilter(
            'always',
            ConvergenceWarning
        )

        start_time = perf_counter()

        candidate_model.fit(
            x_train_tfidf,
            y_train
        )

        training_seconds = (
            perf_counter() - start_time
        )

    start_time = perf_counter()

    candidate_predictions = (
        candidate_model.predict(
            x_validation_tfidf
        )
    )

    prediction_seconds = (
        perf_counter() - start_time
    )

    warning_detected = any(
        issubclass(
            warning.category,
            ConvergenceWarning
        )
        for warning in candidate_warnings
    )

    svc_candidate_models[c_value] = (
        candidate_model
    )

    svc_candidate_predictions[c_value] = (
        candidate_predictions
    )

    svc_tuning_rows.append(
        {
            'c_value': c_value,
            'accuracy': accuracy_score(
                y_validation,
                candidate_predictions
            ),
            'balanced_accuracy': (
                balanced_accuracy_score(
                    y_validation,
                    candidate_predictions
                )
            ),
            'macro_f1': f1_score(
                y_validation,
                candidate_predictions,
                average='macro',
                zero_division=0
            ),
            'weighted_f1': f1_score(
                y_validation,
                candidate_predictions,
                average='weighted',
                zero_division=0
            ),
            'negative_recall': recall_score(
                y_validation,
                candidate_predictions,
                labels=['Negative'],
                average=None,
                zero_division=0
            )[0],
            'iterations': int(
                np.max(
                    np.atleast_1d(
                        candidate_model.n_iter_
                    )
                )
            ),
            'training_seconds': (
                training_seconds
            ),
            'prediction_seconds': (
                prediction_seconds
            ),
            'converged': (
                not warning_detected
            )
        }
    )

# Build the complete comparison table
svc_tuning_table = (
    pd.DataFrame(svc_tuning_rows)
    .drop_duplicates(
        subset='c_value',
        keep='last'
    )
    .sort_values(
        by=[
            'macro_f1',
            'negative_recall'
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    svc_tuning_table.round(4)
)

,c_value,accuracy,balanced_accuracy,macro_f1,weighted_f1,negative_recall,iterations,training_seconds,prediction_seconds,converged
0,0.0500,0.9076,0.8199,0.8118,0.9086,0.8818,11,33.7559,0.0484,True
1,0.1000,0.9072,0.8206,0.8116,0.9085,0.8807,13,34.3952,0.0522,True
2,0.2000,0.9059,0.8190,0.8094,0.9074,0.8764,13,42.1419,0.0510,True
3,0.0300,0.9065,0.8163,0.8090,0.9073,0.8794,11,33.6977,0.0506,True
4,0.5000,0.9033,0.8150,0.8047,0.9051,0.8705,18,54.9741,0.0498,True
5,0.0100,0.9034,0.8065,0.8015,0.9035,0.8746,9,25.7288,0.0500,True
6,1.0000,0.9007,0.8113,0.8004,0.9028,0.8659,17,62.2853,0.0500,True
7,2.0000,0.8980,0.8078,0.7961,0.9004,0.8595,18,64.0552,0.0522,True


In [19]:
# Select the best configuration
selected_svc_row = svc_tuning_table.iloc[0]

selected_svc_c = float(
    selected_svc_row['c_value']
)

selected_svc_model = (
    svc_candidate_models[selected_svc_c]
)

selected_svc_validation_predictions = (
    svc_candidate_predictions[selected_svc_c]
)

selected_svc_training_seconds = float(
    selected_svc_row['training_seconds']
)

selected_svc_prediction_seconds = float(
    selected_svc_row['prediction_seconds']
)

# Show the selected configuration
selected_svc_summary = pd.Series(
    {
        'model': 'LinearSVC',
        'c_value': selected_svc_c,
        'training_seconds': (
            selected_svc_training_seconds
        ),
        'prediction_seconds': (
            selected_svc_prediction_seconds
        ),
        'converged': bool(
            selected_svc_row['converged']
        )
    },
    name='result'
)

display(
    selected_svc_summary.to_frame()
)

,result
model,LinearSVC
c_value,0.0500
training_seconds,33.7559
prediction_seconds,0.0484
converged,True


In [20]:
(
    selected_svc_metrics,
    selected_svc_report,
    selected_svc_confusion
) = evaluate_predictions(
    model_name='linear_svc_tuned_c_0_05',
    y_true=y_validation,
    y_predicted=(
        selected_svc_validation_predictions
    ),
    training_seconds=(
        selected_svc_training_seconds
    )
)

display(selected_svc_metrics)
display(selected_svc_report)
display(selected_svc_confusion)

,value
accuracy,0.9076
balanced_accuracy,0.8199
macro_f1,0.8118
weighted_f1,0.9086
negative_recall,0.8818
training_seconds,33.7559


,precision,recall,f1-score,support
Negative,0.8437,0.8818,0.8623,15335.0000
Neutral,0.6028,0.6246,0.6135,12623.0000
Positive,0.9659,0.9532,0.9595,87050.0000
accuracy,0.9076,0.9076,0.9076,0.9076
macro avg,0.8042,0.8199,0.8118,115008.0000
weighted avg,0.9098,0.9076,0.9086,115008.0000


,Predicted: Negative,Predicted: Neutral,Predicted: Positive
Real: Negative,13522,1604,209
Real: Neutral,2023,7884,2716
Real: Positive,482,3590,82978


In [21]:
validation_model_comparison = (
    pd.DataFrame(experiment_results)
    .T
    .sort_values(
        by=[
            'macro_f1',
            'negative_recall'
        ],
        ascending=False
    )
    .round(4)
)

display(validation_model_comparison)

,accuracy,balanced_accuracy,macro_f1,weighted_f1,negative_recall,training_seconds
linear_svc_tuned_c_0_05,0.9076,0.8199,0.8118,0.9086,0.8818,33.7559
linear_svc_tfidf,0.9007,0.8113,0.8004,0.9028,0.8659,62.9298
logistic_regression_tfidf,0.8820,0.8260,0.7890,0.8900,0.8889,377.0120
dummy_most_frequent,0.7569,0.3333,0.2872,0.6522,0.0000,0.2331


In [22]:
# Build the development set
development_mask = (
    modeling_corpus['dataset_split']
    .isin([
        'train',
        'validation'
    ])
)

test_mask = (
    modeling_corpus['dataset_split']
    .eq('test')
)

x_development = modeling_corpus.loc[
    development_mask,
    'review_text'
]

y_development = modeling_corpus.loc[
    development_mask,
    'sentiment'
]

x_test = modeling_corpus.loc[
    test_mask,
    'review_text'
]

y_test = modeling_corpus.loc[
    test_mask,
    'sentiment'
]

development_restaurants = set(
    modeling_corpus.loc[
        development_mask,
        'item_id'
    ]
)

test_restaurants = set(
    modeling_corpus.loc[
        test_mask,
        'item_id'
    ]
)

final_partition_validation = pd.Series(
    {
        'development_reviews': (
            len(x_development)
        ),
        'test_reviews': len(x_test),
        'development_restaurants': (
            len(development_restaurants)
        ),
        'test_restaurants': (
            len(test_restaurants)
        ),
        'restaurant_overlap': len(
            development_restaurants
            .intersection(test_restaurants)
        ),
        'development_missing_text': (
            x_development.isna().sum()
        ),
        'test_missing_text': (
            x_test.isna().sum()
        )
    },
    name='result'
)

display(
    final_partition_validation
    .to_frame()
)

,result
development_reviews,1046888
test_reviews,110912
development_restaurants,9060
test_restaurants,996
restaurant_overlap,0
development_missing_text,0
test_missing_text,0


In [23]:
# Create the final vectorizer
final_tfidf_vectorizer = clone(
    tfidf_vectorizer
)

# Free the previous matrices
for matrix_name in [
    'x_train_tfidf',
    'x_validation_tfidf'
]:
    globals().pop(
        matrix_name,
        None
    )

gc.collect()

# Fit TF-IDF on training and validation data
start_time = perf_counter()

x_development_tfidf = (
    final_tfidf_vectorizer
    .fit_transform(x_development)
)

final_tfidf_fit_seconds = (
    perf_counter() - start_time
)

# Transform the test set without refitting the vocabulary
start_time = perf_counter()

x_test_tfidf = (
    final_tfidf_vectorizer
    .transform(x_test)
)

final_test_transform_seconds = (
    perf_counter() - start_time
)

In [24]:
# Create the final model
final_svc_model = LinearSVC(
    C=selected_svc_c,
    penalty='l2',
    loss='squared_hinge',
    class_weight='balanced',
    dual='auto',
    max_iter=5_000,
    tol=1e-3,
    random_state=RANDOM_STATE
)

with warnings.catch_warnings(
    record=True
) as final_model_warnings:

    warnings.simplefilter(
        'always',
        ConvergenceWarning
    )

    start_time = perf_counter()

    final_svc_model.fit(
        x_development_tfidf,
        y_development
    )

    final_model_training_seconds = (
        perf_counter() - start_time
    )

final_warning_detected = any(
    issubclass(
        warning.category,
        ConvergenceWarning
    )
    for warning in final_model_warnings
)

# Evaluate on the test set once
start_time = perf_counter()

final_test_predictions = (
    final_svc_model.predict(
        x_test_tfidf
    )
)

final_test_prediction_seconds = (
    perf_counter() - start_time
)

final_model_iterations = int(
    np.max(
        np.atleast_1d(
            final_svc_model.n_iter_
        )
    )
)

In [25]:
final_test_metrics = pd.Series(
    {
        'accuracy': accuracy_score(
            y_test,
            final_test_predictions
        ),
        'balanced_accuracy': (
            balanced_accuracy_score(
                y_test,
                final_test_predictions
            )
        ),
        'macro_f1': f1_score(
            y_test,
            final_test_predictions,
            average='macro',
            zero_division=0
        ),
        'weighted_f1': f1_score(
            y_test,
            final_test_predictions,
            average='weighted',
            zero_division=0
        ),
        'negative_recall': recall_score(
            y_test,
            final_test_predictions,
            labels=['Negative'],
            average=None,
            zero_division=0
        )[0]
    },
    name='final_test'
).to_frame('value').round(4)

final_test_report = (
    pd.DataFrame(
        classification_report(
            y_test,
            final_test_predictions,
            labels=sentiment_order,
            output_dict=True,
            zero_division=0
        )
    )
    .T
    .round(4)
)

final_test_confusion = pd.DataFrame(
    confusion_matrix(
        y_test,
        final_test_predictions,
        labels=sentiment_order
    ),
    index=[
        f'Actual: {label}'
        for label in sentiment_order
    ],
    columns=[
        f'Predicted: {label}'
        for label in sentiment_order
    ]
)

final_training_summary = pd.Series(
    {
        'development_rows': (
            len(x_development)
        ),
        'test_rows': len(x_test),
        'tfidf_features': (
            x_development_tfidf.shape[1]
        ),
        'tfidf_fit_seconds': (
            final_tfidf_fit_seconds
        ),
        'test_transform_seconds': (
            final_test_transform_seconds
        ),
        'model_iterations': (
            final_model_iterations
        ),
        'model_training_seconds': (
            final_model_training_seconds
        ),
        'test_prediction_seconds': (
            final_test_prediction_seconds
        ),
        'converged': (
            not final_warning_detected
        )
    },
    name='result'
).to_frame().round(4)

display(final_training_summary)
display(final_test_metrics)
display(final_test_report)
display(final_test_confusion)

,result
development_rows,1046888
test_rows,110912
tfidf_features,60000
tfidf_fit_seconds,59.2410
test_transform_seconds,4.5802
model_iterations,12
model_training_seconds,46.5120
test_prediction_seconds,0.0530
converged,True


,value
accuracy,0.9048
balanced_accuracy,0.8164
macro_f1,0.8078
weighted_f1,0.9060
negative_recall,0.8793


,precision,recall,f1-score,support
Negative,0.8448,0.8793,0.8617,15136.0000
Neutral,0.5893,0.6190,0.6038,12154.0000
Positive,0.9651,0.9509,0.9580,83622.0000
accuracy,0.9048,0.9048,0.9048,0.9048
macro avg,0.7998,0.8164,0.8078,110912.0000
weighted avg,0.9075,0.9048,0.9060,110912.0000


,Predicted: Negative,Predicted: Neutral,Predicted: Positive
Actual: Negative,13309,1623,204
Actual: Neutral,1960,7523,2671
Actual: Positive,485,3619,79518


### Check results by language

English interface text does not mean the model was trained on English reviews only. Report both Spanish and English performance on the untouched test set. The split is still based on restaurant IDs.

In [26]:
language_test = modeling_corpus.loc[test_mask, ['language', 'item_id']].copy()
language_test['actual'] = y_test.to_numpy()
language_test['predicted'] = final_test_predictions
language_metrics = []
for language, subset in language_test.groupby('language', observed=True):
    language_metrics.append({
        'language': language,
        'reviews': len(subset),
        'restaurants': subset['item_id'].nunique(),
        'macro_f1': f1_score(subset['actual'], subset['predicted'],
                             labels=sentiment_order, average='macro', zero_division=0),
        'negative_recall': recall_score(subset['actual'], subset['predicted'],
                                        labels=['Negative'], average=None, zero_division=0)[0]
    })
language_metrics = pd.DataFrame(language_metrics).sort_values('language')
display(language_metrics.round(4))


,language,reviews,restaurants,macro_f1,negative_recall
0,en,19777,625,0.7928,0.8508
1,es,91135,987,0.8092,0.8834


## 10. Save the reproducible model and test results


In [27]:
# Create output folders
MODELS_PATH = (
    ARTIFACTS_PATH / 'models'
)

METRICS_PATH = (
    ARTIFACTS_PATH / 'metrics'
)

MODELS_PATH.mkdir(
    parents=True,
    exist_ok=True
)

METRICS_PATH.mkdir(
    parents=True,
    exist_ok=True
)

# Combine the vectorizer and classifier
final_sentiment_pipeline = Pipeline(
    steps=[
        (
            'tfidf',
            final_tfidf_vectorizer
        ),
        (
            'classifier',
            final_svc_model
        )
    ]
)

# Define output paths
pipeline_path = (
    MODELS_PATH
    / 'sentiment_tfidf_linear_svc.joblib'
)

metadata_path = (
    MODELS_PATH
    / 'sentiment_model_metadata.json'
)

test_metrics_path = (
    METRICS_PATH
    / 'final_test_metrics.csv'
)

classification_report_path = (
    METRICS_PATH
    / 'final_test_classification_report.csv'
)

confusion_matrix_path = (
    METRICS_PATH
    / 'final_test_confusion_matrix.csv'
)

# Save the pipeline
joblib.dump(
    final_sentiment_pipeline,
    pipeline_path,
    compress=3
)

# Save evaluation tables
final_test_metrics.to_csv(
    test_metrics_path
)

final_test_report.to_csv(
    classification_report_path
)

final_test_confusion.to_csv(
    confusion_matrix_path
)

In [28]:
test_metrics_dictionary = {
    metric: float(value)
    for metric, value
    in final_test_metrics['value'].items()
}

model_metadata = {
    'model_name': (
        'sentiment_tfidf_linear_svc'
    ),
    'model_version': '1.0.0',
    'created_at_utc': (
        datetime.now(timezone.utc)
        .isoformat()
    ),
    'predictor': 'review_text',
    'target': 'sentiment',
    'classes': (
        final_svc_model
        .classes_
        .tolist()
    ),
    'model_parameters': {
        'algorithm': 'LinearSVC',
        'c_value': selected_svc_c,
        'penalty': 'l2',
        'loss': 'squared_hinge',
        'class_weight': 'balanced',
        'max_iter': 5_000,
        'tol': 1e-3,
        'random_state': RANDOM_STATE
    },
    'tfidf_parameters': {
        'ngram_range': [1, 2],
        'min_df': 10,
        'max_df': 0.98,
        'max_features': 60_000,
        'sublinear_tf': True,
        'norm': 'l2'
    },
    'evaluation_design': {
        'group_variable': 'item_id',
        'development_reviews': int(
            len(x_development)
        ),
        'test_reviews': int(
            len(x_test)
        ),
        'development_restaurants': int(
            len(development_restaurants)
        ),
        'test_restaurants': int(
            len(test_restaurants)
        ),
        'restaurant_overlap': 0
    },
    'test_metrics': (
        test_metrics_dictionary
    )
}

with metadata_path.open(
    mode='w',
    encoding='utf-8'
) as metadata_file:

    json.dump(
        model_metadata,
        metadata_file,
        ensure_ascii=False,
        indent=4
    )

In [29]:
# Reload the saved pipeline
loaded_sentiment_pipeline = joblib.load(
    pipeline_path
)

# Check predictions on a sample of 1,000 reviews
verification_texts = x_test.iloc[
    :1_000
]

loaded_predictions = (
    loaded_sentiment_pipeline.predict(
        verification_texts
    )
)

expected_predictions = (
    final_test_predictions[:1_000]
)

predictions_match = np.array_equal(
    loaded_predictions,
    expected_predictions
)

artifact_paths = {
    'pipeline': pipeline_path,
    'metadata': metadata_path,
    'test_metrics': test_metrics_path,
    'classification_report': (
        classification_report_path
    ),
    'confusion_matrix': (
        confusion_matrix_path
    )
}

artifact_validation = pd.DataFrame(
    [
        {
            'artifact': artifact_name,
            'file': artifact_path.name,
            'exists': artifact_path.exists(),
            'size_mb': (
                artifact_path.stat().st_size
                / 1024 ** 2
            )
        }
        for artifact_name, artifact_path
        in artifact_paths.items()
    ]
)

print('Predictions match:', predictions_match)

display(artifact_validation.round(4))

Predictions match: True


,artifact,file,exists,size_mb
0,pipeline,sentiment_tfidf_linear_svc.joblib,True,2.0418
1,metadata,sentiment_model_metadata.json,True,0.0011
2,test_metrics,final_test_metrics.csv,True,0.0001
3,classification_report,final_test_classification_report.csv,True,0.0003
4,confusion_matrix,final_test_confusion_matrix.csv,True,0.0001


In [30]:
# Save model selection results
svc_tuning_table.to_csv(
    METRICS_PATH / 'svc_tuning_results.csv',
    index=False
)

validation_model_comparison.to_csv(
    METRICS_PATH / 'validation_model_comparison.csv'
)

selected_svc_metrics.to_csv(
    METRICS_PATH / 'selected_validation_metrics.csv'
)

selected_svc_report.to_csv(
    METRICS_PATH / 'selected_validation_classification_report.csv'
)

selected_svc_confusion.to_csv(
    METRICS_PATH / 'selected_validation_confusion_matrix.csv'
)

print(
    'Validation results saved successfully'
)

Validation results saved successfully


In [31]:
language_metrics.to_csv(METRICS_PATH / 'final_test_by_language.csv', index=False)
print('Saved test breakdown by language.')


Saved test breakdown by language.


## 11. Track experiments with MLflow


In [32]:
print(
    'MLflow:',
    mlflow.__version__
)

print(
    'scikit-learn:',
    sklearn.__version__
)


MLflow: 3.11.1
scikit-learn: 1.6.1


### Configure the SQLite tracking store


In [33]:
# Define the metadata store
MLFLOW_DATABASE_PATH = (
    ARTIFACTS_PATH
    / 'mlflow.db'
).resolve()

# Define artifact storage
MLFLOW_ARTIFACTS_PATH = (
    ARTIFACTS_PATH
    / 'mlflow_artifacts'
).resolve()

MLFLOW_ARTIFACTS_PATH.mkdir(
    parents=True,
    exist_ok=True
)

# Build the local SQLite connection
mlflow_tracking_uri = (
    'sqlite:///'
    + MLFLOW_DATABASE_PATH.as_posix()
)

mlflow.set_tracking_uri(
    mlflow_tracking_uri
)

mlflow_client = MlflowClient(
    tracking_uri=mlflow_tracking_uri
)

experiment_name = (
    'restaurant_sentiment_classification'
)

# Create the experiment with an explicit artifact location
existing_experiment = (
    mlflow.get_experiment_by_name(
        experiment_name
    )
)

if existing_experiment is None:

    mlflow_client.create_experiment(
        name=experiment_name,
        artifact_location=(
            MLFLOW_ARTIFACTS_PATH.as_uri()
        )
    )

mlflow_experiment = mlflow.set_experiment(
    experiment_name
)

mlflow_configuration = pd.Series(
    {
        'mlflow_version': (
            mlflow.__version__
        ),
        'sklearn_version': (
            sklearn.__version__
        ),
        'tracking_backend': 'SQLite',
        'tracking_uri': (
            mlflow.get_tracking_uri()
        ),
        'artifact_location': (
            mlflow_experiment
            .artifact_location
        ),
        'experiment_name': (
            mlflow_experiment.name
        ),
        'experiment_id': (
            mlflow_experiment
            .experiment_id
        )
    },
    name='result'
)

display(
    mlflow_configuration.to_frame()
)

,result
mlflow_version,3.11.1
sklearn_version,1.6.1
tracking_backend,SQLite
tracking_uri,sqlite:////Users/carolina/1.Restaurant_intelli...
artifact_location,file:///Users/carolina/1.Restaurant_intelligen...
experiment_name,restaurant_sentiment_classification
experiment_id,1


### Register baseline runs


In [34]:
# Get the experiment ID
experiment_id = str(
    mlflow_experiment.experiment_id
)

# Define baseline model parameters
baseline_parameters = {
    'dummy_most_frequent': {
        'algorithm': 'DummyClassifier',
        'strategy': 'most_frequent'
    },
    'logistic_regression_tfidf': {
        'algorithm': 'LogisticRegression',
        'c_value': 1.0,
        'solver': 'saga',
        'class_weight': 'balanced',
        'converged': not convergence_warning_detected
    }
}

# Log baseline models
for model_name, model_parameters in (
    baseline_parameters.items()
):

    with mlflow.start_run(
        experiment_id=experiment_id,
        run_name=model_name,
        tags={
            'stage': 'validation',
            'task': (
                'sentiment_classification'
            )
        }
    ):

        mlflow.log_params(
            model_parameters
        )

        mlflow.log_metrics(
            {
                f'validation_{metric_name}': (
                    float(metric_value)
                )
                for metric_name, metric_value
                in experiment_results[
                    model_name
                ].items()
            }
        )

print(
    'Baseline models logged'
)

Baseline models logged


In [35]:
# Get the baseline runs
baseline_runs = mlflow.search_runs(
    experiment_ids=[
        experiment_id
    ],
    filter_string=(
        "tags.stage = 'validation'"
    )
)

baseline_run_summary = baseline_runs[
    [
        'tags.mlflow.runName',
        'metrics.validation_accuracy',
        'metrics.validation_macro_f1',
        'metrics.validation_negative_recall',
        'metrics.validation_training_seconds'
    ]
].sort_values(
    by='metrics.validation_macro_f1',
    ascending=False
)

display(
    baseline_run_summary.round(4)
)

print(
    'Runs logged:',
    len(baseline_runs)
)

,tags.mlflow.runName,metrics.validation_accuracy,metrics.validation_macro_f1,metrics.validation_negative_recall,metrics.validation_training_seconds
0,logistic_regression_tfidf,0.8820,0.7890,0.8889,377.0120
2,logistic_regression_tfidf,0.8820,0.7890,0.8889,389.6705
1,dummy_most_frequent,0.7569,0.2872,0.0000,0.2331
3,dummy_most_frequent,0.7569,0.2872,0.0000,0.2518


Runs logged: 4


### Register tuning runs


In [36]:
# Get existing run names
existing_runs = mlflow.search_runs(
    experiment_ids=[
        experiment_id
    ]
)

existing_run_names = set(
    existing_runs[
        'tags.mlflow.runName'
    ]
    .dropna()
    .tolist()
)

# Log every value of C
for _, tuning_row in (
    svc_tuning_table
    .sort_values('c_value')
    .iterrows()
):

    c_value = float(
        tuning_row['c_value']
    )

    run_name = (
        'linear_svc_c_'
        + str(c_value).replace(
            '.',
            '_'
        )
    )

    # Skip duplicate runs
    if run_name in existing_run_names:
        continue

    selected_candidate = bool(
        np.isclose(
            c_value,
            selected_svc_c
        )
    )

    with mlflow.start_run(
        experiment_id=experiment_id,
        run_name=run_name,
        tags={
            'stage': (
                'hyperparameter_tuning'
            ),
            'task': (
                'sentiment_classification'
            ),
            'selected': str(
                selected_candidate
            ).lower()
        }
    ):

        mlflow.log_params(
            {
                'algorithm': 'LinearSVC',
                'c_value': c_value,
                'penalty': 'l2',
                'loss': 'squared_hinge',
                'class_weight': 'balanced',
                'max_iter': 5_000,
                'tol': 1e-3,
                'tfidf_features': 60_000,
                'converged': bool(
                    tuning_row['converged']
                )
            }
        )

        mlflow.log_metrics(
            {
                'validation_accuracy': float(
                    tuning_row['accuracy']
                ),
                'validation_balanced_accuracy': float(
                    tuning_row[
                        'balanced_accuracy'
                    ]
                ),
                'validation_macro_f1': float(
                    tuning_row['macro_f1']
                ),
                'validation_weighted_f1': float(
                    tuning_row['weighted_f1']
                ),
                'validation_negative_recall': float(
                    tuning_row[
                        'negative_recall'
                    ]
                ),
                'training_iterations': float(
                    tuning_row['iterations']
                ),
                'training_seconds': float(
                    tuning_row[
                        'training_seconds'
                    ]
                ),
                'prediction_seconds': float(
                    tuning_row[
                        'prediction_seconds'
                    ]
                )
            }
        )

    existing_run_names.add(
        run_name
    )

print(
    'LinearSVC configurations logged'
)

LinearSVC configurations logged


In [37]:
all_mlflow_runs = mlflow.search_runs(
    experiment_ids=[
        experiment_id
    ]
)

svc_mlflow_runs = all_mlflow_runs.loc[
    all_mlflow_runs[
        'tags.stage'
    ].eq(
        'hyperparameter_tuning'
    )
]

svc_mlflow_summary = (
    svc_mlflow_runs[
        [
            'tags.mlflow.runName',
            'params.c_value',
            'metrics.validation_macro_f1',
            'metrics.validation_negative_recall',
            'metrics.training_seconds',
            'tags.selected'
        ]
    ]
    .sort_values(
        by='metrics.validation_macro_f1',
        ascending=False
    )
)

display(
    svc_mlflow_summary
    .round(4)
)

print(
    'Runs de LinearSVC:',
    len(svc_mlflow_runs)
)

print(
    'Runs totales:',
    len(all_mlflow_runs)
)

,tags.mlflow.runName,params.c_value,metrics.validation_macro_f1,metrics.validation_negative_recall,metrics.training_seconds,tags.selected
8,linear_svc_c_0_05,0.05,0.8118,0.8818,32.2528,true
7,linear_svc_c_0_1,0.1,0.8116,0.8807,37.0834,false
6,linear_svc_c_0_2,0.2,0.8094,0.8764,44.7554,false
9,linear_svc_c_0_03,0.03,0.8090,0.8794,31.3206,false
5,linear_svc_c_0_5,0.5,0.8047,0.8705,52.6055,false
10,linear_svc_c_0_01,0.01,0.8015,0.8746,26.8707,false
4,linear_svc_c_1_0,1.0,0.8004,0.8659,62.7708,false
3,linear_svc_c_2_0,2.0,0.7961,0.8595,65.5824,false


Runs de LinearSVC: 8
Runs totales: 13


### Log the final trained pipeline


In [38]:
# Create examples for the model input
mlflow_input_example = np.array(
    [
        'La comida estuvo excelente',
        'El servicio fue lento y malo',
        'The restaurant was acceptable'
    ],
    dtype=str
)

mlflow_example_predictions = (
    final_sentiment_pipeline.predict(
        mlflow_input_example
    )
)

# Infer the input and output signature
mlflow_model_signature = infer_signature(
    mlflow_input_example,
    mlflow_example_predictions
)

# Prepare evaluation metrics
validation_metrics_to_log = {
    f'validation_{metric_name}': float(
        metric_value
    )
    for metric_name, metric_value
    in selected_svc_metrics['value'].items()
}

test_metrics_to_log = {
    f'test_{metric_name}': float(
        metric_value
    )
    for metric_name, metric_value
    in final_test_metrics['value'].items()
}

# Log the final run
with mlflow.start_run(
    experiment_id=experiment_id,
    run_name=f'final_linear_svc_c_{selected_svc_c:g}',
    tags={
        'stage': 'final_test',
        'task': 'sentiment_classification',
        'selected_model': 'true',
        'evaluation_unit': (
            'unseen_restaurants'
        )
    }
) as final_mlflow_run:

    mlflow.log_params(
        {
            'algorithm': 'LinearSVC',
            'c_value': selected_svc_c,
            'penalty': 'l2',
            'loss': 'squared_hinge',
            'class_weight': 'balanced',
            'tfidf_features': 60_000,
            'ngram_range': '1-2',
            'development_reviews': int(
                len(x_development)
            ),
            'test_reviews': int(
                len(x_test)
            ),
            'test_restaurants': int(
                len(test_restaurants)
            ),
            'restaurant_overlap': 0
        }
    )

    mlflow.log_metrics(
        validation_metrics_to_log
    )

    mlflow.log_metrics(
        test_metrics_to_log
    )

    mlflow.log_metrics(
        {
            'final_tfidf_fit_seconds': float(
                final_tfidf_fit_seconds
            ),
            'final_model_training_seconds': float(
                final_model_training_seconds
            ),
            'final_test_prediction_seconds': float(
                final_test_prediction_seconds
            )
        }
    )

    # Log evaluation files
    mlflow.log_artifacts(
        str(METRICS_PATH),
        artifact_path='evaluation'
    )

    mlflow.log_artifact(
        str(metadata_path),
        artifact_path='metadata'
    )

    mlflow.log_artifact(
        str(pipeline_path),
        artifact_path='joblib'
    )

    mlflow.log_artifact(
        str(project_config_path),
        artifact_path='configuration'
    )

    # Log the pipeline as an MLflow model
    mlflow_model_information = (
        mlflow.sklearn.log_model(
            sk_model=(
                final_sentiment_pipeline
            ),
            name='sentiment_pipeline',
            signature=(
                mlflow_model_signature
            ),
            input_example=(
                mlflow_input_example
            ),
            serialization_format=(
                'cloudpickle'
            )
        )
    )

final_mlflow_run_id = (
    final_mlflow_run.info.run_id
)

print(
    'Final run:',
    final_mlflow_run_id
)

print(
    'Model:',
    mlflow_model_information.model_uri
)

2026/09/23 23:08:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Final run: dd278a627e6a407e8d4cde3bb3b360af
Model: models:/m-8a7d8fb3fba04a2883f5602165c7a590


In [39]:
# Get all runs
all_mlflow_runs = mlflow.search_runs(
    experiment_ids=[
        experiment_id
    ]
)

# Load the logged model
mlflow_loaded_model = (
    mlflow.sklearn.load_model(
        mlflow_model_information.model_uri
    )
)

# Check its predictions
mlflow_loaded_predictions = (
    mlflow_loaded_model.predict(
        mlflow_input_example
    )
)

mlflow_predictions_match = (
    np.array_equal(
        mlflow_loaded_predictions,
        mlflow_example_predictions
    )
)

final_run_record = all_mlflow_runs.loc[
    all_mlflow_runs['run_id'].eq(
        final_mlflow_run_id
    )
]

mlflow_final_validation = pd.Series(
    {
        'registered_runs': (
            len(all_mlflow_runs)
        ),
        'final_run_status': (
            final_run_record.iloc[0][
                'status'
            ]
        ),
        'final_run_id': (
            final_mlflow_run_id
        ),
        'model_uri': (
            mlflow_model_information
            .model_uri
        ),
        'loaded_predictions_match': (
            mlflow_predictions_match
        )
    },
    name='result'
)

display(
    mlflow_final_validation.to_frame()
)

,result
registered_runs,14
final_run_status,FINISHED
final_run_id,dd278a627e6a407e8d4cde3bb3b360af
model_uri,models:/m-8a7d8fb3fba04a2883f5602165c7a590
loaded_predictions_match,True


### Register the selected model with a champion alias


In [40]:
# Register the final model
registered_model_name = (
    'restaurant_sentiment_classifier'
)

registered_model_alias = 'champion'

registered_model_version = mlflow.register_model(
    model_uri=(
        mlflow_model_information.model_uri
    ),
    name=registered_model_name
)

# Set the model alias
model_registry_client = MlflowClient()

model_registry_client.set_registered_model_alias(
    name=registered_model_name,
    alias=registered_model_alias,
    version=registered_model_version.version
)

# Validate the registered model
champion_model_uri = (
    f'models:/{registered_model_name}'
    f'@{registered_model_alias}'
)

champion_model = mlflow.sklearn.load_model(
    champion_model_uri
)

champion_predictions = champion_model.predict(
    mlflow_input_example
)

champion_predictions_match = np.array_equal(
    champion_predictions,
    mlflow_example_predictions
)

model_registry_validation = pd.Series(
    {
        'registered_model': (
            registered_model_name
        ),
        'version': (
            registered_model_version.version
        ),
        'alias': (
            registered_model_alias
        ),
        'model_uri': (
            champion_model_uri
        ),
        'loaded_predictions_match': (
            champion_predictions_match
        )
    },
    name='result'
)

display(
    model_registry_validation.to_frame()
)

Registered model 'restaurant_sentiment_classifier' already exists. Creating a new version of this model...
Created version '2' of model 'restaurant_sentiment_classifier'.


,result
registered_model,restaurant_sentiment_classifier
version,2
alias,champion
model_uri,models:/restaurant_sentiment_classifier@champion
loaded_predictions_match,True


## 12. Review themes for actionable diagnosis

Use transparent bilingual keyword patterns to summarize what negative reviews mention. These counts are descriptive: a keyword is not a verified cause, and a theme may appear in a negated sentence. Show a theme only after at least five mentions. The data is aggregated before the app loads.

In [41]:
# Summarize recurring topics in historical negative reviews.
import re

aspect_patterns = {
    'Food': r'\b(?:food|meal|dish(?:es)?|taste|flavo[u]?r|comida|platos?|sabor|cocina)\b',
    'Service': r'\b(?:service|staff|waiter|waitress|servicio|atenci[oó]n|camarer[oa]s?|personal)\b',
    'Atmosphere': r'\b(?:atmosphere|ambience|music|terrace|ambiente|m[uú]sica|terraza)\b',
    'Price and value': r'\b(?:price|expensive|cheap|value|precio|car[oa]s?|barat[oa]s?)\b',
    'Waiting time': r'\b(?:wait(?:ing|ed)?|delay|slow|espera|tard[oa]|lent[oa])\b',
    'Cleanliness': r'\b(?:clean|dirty|hygiene|limpi[oa]|suci[oa]|higiene)\b',
}
negative_text = modeling_corpus.loc[
    modeling_corpus['sentiment'].eq('Negative'), ['item_id', 'review_text']
].copy()
negative_counts = negative_text.groupby('item_id', observed=True).size().rename('negative_reviews')
aspect_rows = []
for aspect, pattern in aspect_patterns.items():
    mask = negative_text['review_text'].str.contains(pattern, flags=re.IGNORECASE,
                                                      regex=True, na=False)
    counts = negative_text.loc[mask].groupby('item_id', observed=True).size()
    aspect_rows.append(pd.DataFrame({'item_id': counts.index, 'aspect': aspect,
                                     'mentions': counts.to_numpy()}))
review_aspects = pd.concat(aspect_rows, ignore_index=True).merge(
    negative_counts.reset_index(), on='item_id', validate='many_to_one'
)
review_aspects['share_of_negative_reviews'] = (
    review_aspects['mentions'] / review_aspects['negative_reviews'] * 100
)
aspects_path = DATA_PATH / 'review_aspects.parquet'
review_aspects.to_parquet(aspects_path, index=False)
print(f'Saved {len(review_aspects):,} restaurant and theme pairs to {aspects_path}')


Saved 29,006 restaurant and theme pairs to /Users/carolina/1.Restaurant_intelligence/data/review_aspects.parquet


## 13. Application

The app is stored as regular Python files in `app/` and `streamlit_app.py`. It reads the products of notebook 2 and the pipeline saved above. The repository bundle also includes the app, requirements and README, so notebook execution does not overwrite deployment dependencies. Launch it from the project root with `streamlit run streamlit_app.py`. The optional API is exposed in `app/api.py`.

The interface uses simple English. The visible alert reference date comes from `data/project_config.json`. The app reports missing platform ratings as unavailable, compares a restaurant with other restaurants in its group, and treats LinearSVC margins as scores rather than probabilities.

In [42]:
from pathlib import Path
import ast

ROOT_PATH = Path.cwd().resolve()
required_files = [
    ROOT_PATH / 'streamlit_app.py',
    ROOT_PATH / 'app' / 'ui.py',
    ROOT_PATH / 'app' / 'inference.py',
    ROOT_PATH / 'app' / 'explainability.py',
    ROOT_PATH / 'app' / 'api.py',
    ROOT_PATH / 'requirements.txt',
    ROOT_PATH / 'requirements-modeling.txt',
]
for file_path in required_files:
    if not file_path.is_file():
        raise FileNotFoundError(f'Place the application file in the project root: {file_path}')
    if file_path.suffix == '.py':
        ast.parse(file_path.read_text(encoding='utf-8'))
print('The independent application files are present and have valid syntax.')


The independent application files are present and have valid syntax.


### Run the app or API

- Streamlit: `streamlit run streamlit_app.py`
- Optional API: `uvicorn app.api:app --reload`
- For deployment, copy the app files, `data/*.parquet`, `data/project_config.json`, and `artifacts/models/sentiment_tfidf_linear_svc.joblib` into the repository. Use `requirements.txt` for Streamlit Cloud. Data and model files must be present before the app starts.

**Limitations:** Ratings provide the sentiment ground truth; the app cannot identify service causes from the classifier alone. Alerts are descriptive and historical. The dataset contains Spanish and English reviews; see the test breakdown by language before describing English language performance.